# Add Author and Country information

In [1]:
from Bio import Entrez
import pandas as pd
import datetime
import time

import os
import sys

In [ ]:
# year_start = int(sys.argv[1])
# year_end = int(sys.argv[2])

year_start = 1980
# year_end = 2026 # Atual year used in the final project.
year_end = 1980  # Set the same start and end year for example.

print(f"----- Downloading PubMed data from {year_start} to {year_end} -----")

----- Downloading PubMed data from 1980 to 1980 -----


In [3]:
Entrez.email = "YOUR_EMAIL_HERE"

In [4]:
def get_pubmed_data(yr, month, day, n_days_window, n_papers):
    start_date = datetime.date(yr, month, day)
    if n_days_window == 0:
        end_date = start_date
    else:
        end_date = start_date + datetime.timedelta(days=n_days_window)
    current_date = start_date

    df_out = []
    while current_date <= end_date:
        date_str = current_date.strftime("%Y/%m/%d")
        print(f"Searching for date: {date_str}...")

        query = f'("{date_str}"[Date - Create] : "{date_str}"[Date - Create]) AND (hasabstract) NOT preprint[filter]'
        print(query)

        # Perform the search
        search_handle = Entrez.esearch(
            db="pubmed",
            term=query,
            retmax=n_papers,
        )
        search_results = Entrez.read(search_handle)
        search_handle.close()

        id_list = search_results["IdList"]
        if len(id_list) == 0:
            print(f"No results found for date: {date_str}. Moving to next date.\n")
            current_date += datetime.timedelta(days=1)
            continue
        else:
            print(f"Found {len(id_list)} results. Top 5 IDs: {id_list[:5]} ...\n")

        # Fetch details for the found IDs
        fetch_handle = Entrez.efetch(db="pubmed", id=",".join(id_list), retmode="xml")
        time.sleep(0.2)

        records = Entrez.read(fetch_handle)
        time.sleep(0.2)

        fetch_handle.close()
        time.sleep(0.2)

        for article in records["PubmedArticle"]:
            df_CorAuthors = []

            # Title
            title = article["MedlineCitation"]["Article"]["ArticleTitle"]

            # Affiliations information.
            try:

                for auth in article["MedlineCitation"]["Article"]["AuthorList"][
                    -1:
                ]:  # Only look at the last author
                    name = f"{auth.get('LastName', '')} {auth.get('ForeName', '')}"
                    affiliations = auth.get("AffiliationInfo", [])
                    for aff in affiliations[
                        :1
                    ]:  # Only look at the first affiliation for simplicity
                        aff_text = aff.get("Affiliation", "")
                        df_CorAuthors_tmp = pd.DataFrame(
                            {
                                "PMID": article["MedlineCitation"]["PMID"],
                                "Author": name,
                                "Affiliation": aff_text,
                                "Country": (
                                    aff_text.split(",")[-1].strip()
                                    if "," in aff_text
                                    else ""
                                ),
                            },
                            index=[0],
                        )
                        df_CorAuthors.append(df_CorAuthors_tmp)
                    df_CorAuthors = pd.concat(df_CorAuthors, ignore_index=True)

            except Exception as e:
                df_CorAuthors = pd.DataFrame(
                    {
                        "PMID": article["MedlineCitation"]["PMID"],
                        "Author": None,
                        "Affiliation": None,
                        "Country": None,
                    },
                    index=[0],
                )

            # Abstract
            abstract = (
                article["MedlineCitation"]["Article"]
                .get("Abstract", {})
                .get("AbstractText", [])
            )
            if abstract:
                abstract = [str(x) for x in abstract]
                if len(abstract) > 1:
                    abstract2 = "".join(abstract)
                else:
                    abstract2 = abstract[0]
            else:
                abstract2 = "No abstract available."

            df_out_tmp = pd.DataFrame(
                {
                    "Date": date_str,
                    "Year": date_str.split("/")[0],
                    "PMID": article["MedlineCitation"]["PMID"],
                    "title": title,
                    "abstract": abstract2,
                    "journal": article["MedlineCitation"]["Article"]["Journal"][
                        "Title"
                    ],
                    "corresponding_authors": df_CorAuthors.Author.tolist(),
                    "corresponding_affiliations": df_CorAuthors.Affiliation.tolist(),
                    "corresponding_countries": df_CorAuthors.Country.tolist(),
                },
                index=[0],
            )
            df_out.append(df_out_tmp)

        # Update date for next iteration
        current_date += datetime.timedelta(days=1)
        date_str = current_date.strftime("%Y/%m/%d")

    if len(df_out) > 0:
        df_out = pd.concat(df_out, ignore_index=True)
    else:
        df_out = pd.DataFrame(
            columns=["Date", "Year", "PMID", "title", "abstract", "journal"]
        )

    return df_out

In [5]:
# Parse finished years.
DIR = "./"
cmd = f"mkdir -p {DIR}/DataCollection/"
os.system(cmd)

years_finished = os.listdir(f"{DIR}/DataCollection/")
years_finished = [year for year in years_finished if year.startswith("pubmed_data_")]
years_finished = [int(year.split("_")[-1].split(".")[0]) for year in years_finished]
years_finished.sort()
print(years_finished[:5])
print(years_finished[-5:])

[1981, 1982, 1983, 1984, 1985]
[2022, 2023, 2024, 2025, 2026]


In [ ]:
# day = 1
n_days_window = 0
# n_papers = 10000 # Actual number used for the final project.
n_papers = 100  # Set a small number for example.

dict_day_by_month = {
    1: 31,
    2: 28,  # Ignoring leap years for simplicity
    3: 31,
    4: 30,
    5: 31,
    6: 30,
    7: 31,
    8: 31,
    9: 30,
    10: 31,
    11: 30,
    12: 31,
}

In [ ]:
for year in range(year_start, year_end + 1):
    if year in years_finished:
        print(f"Year {year} already processed. Skipping...")
        continue

    df_out = []
    print(f">>>>> Processing year: {year}...")

    for month in range(1, 13):
        for day in range(1, dict_day_by_month[month] + 1):
            # try up to 3 times if there is a runtime error.
            attemps = 0
            while attemps < 3:
                try:
                    df_out.append(
                        get_pubmed_data(year, month, day, n_days_window, n_papers)
                    )
                    break  # If successful, break out of the retry loop
                except Exception as e:
                    attemps += 1
                    print(f"Error occurred: {e}. Attempt {attemps}/3. Retrying...")
                    time.sleep(5)  # Wait before retrying

    # Save output for the year.
    df_out = pd.concat(df_out, axis=0, ignore_index=True)
    df_out.to_csv(f"{DIR}/DataCollection/pubmed_data_{year}.tsv", index=False, sep="\t")

>>>>> Processing year: 1980...
Searching for date: 1980/01/01...
("1980/01/01"[Date - Create] : "1980/01/01"[Date - Create]) AND (hasabstract) NOT preprint[filter]
Found 100 results. Top 5 IDs: ['25121255', '25032442', '18962620', '18962619', '18962618'] ...

Searching for date: 1980/02/01...
("1980/02/01"[Date - Create] : "1980/02/01"[Date - Create]) AND (hasabstract) NOT preprint[filter]
Found 100 results. Top 5 IDs: ['18962643', '18962642', '18962641', '18962640', '18962639'] ...

Searching for date: 1980/03/01...
("1980/03/01"[Date - Create] : "1980/03/01"[Date - Create]) AND (hasabstract) NOT preprint[filter]
Found 100 results. Top 5 IDs: ['18962668', '18962667', '18962666', '18962665', '18962664'] ...

Searching for date: 1980/04/01...
("1980/04/01"[Date - Create] : "1980/04/01"[Date - Create]) AND (hasabstract) NOT preprint[filter]
Found 100 results. Top 5 IDs: ['18962688', '18962687', '18962686', '18962685', '18962684'] ...

Searching for date: 1980/05/01...
("1980/05/01"[Date 

In [8]:
df_out.head()

,Date,Year,PMID,title,abstract,journal,corresponding_authors,corresponding_affiliations,corresponding_countries
0,1980/01/01,1980,18962620,Complex formation between aluminium and 3-hydr...,The formation of chelates between aluminium(II...,Talanta,None,None,None
1,1980/01/01,1980,18962619,"The protonation and dimerization of 2,4-, 2,5-...","The protonation equilibria of 2,4-, 2,5-, 2,6-...",Talanta,None,None,None
2,1980/01/01,1980,18962618,Routine accurate determination of silica in si...,An accurate method for routine determination o...,Talanta,None,None,None
3,1980/01/01,1980,18962617,New turbidimetric method for determination of ...,Sulphate in the range (1-5) x 10(-5)M is deter...,Talanta,None,None,None
4,1980/01/01,1980,18962616,Arsenazo III as a spectrophotometric reagent f...,Arsenazo III is proposed as a spectrophotometr...,Talanta,None,None,None
